# 00 · Data Inventory

**Purpose.** Обзор всех доступных данных: файлы, размеры, диапазоны дат, колонки, whitelist, missingness и частота обновления (daily / event-driven / monthly).

**What to look for.**
- какие файлы в data/processed и их размеры
- диапазоны дат и число строк по основным датасетам
- какие 26 whitelist-фич реально присутствуют в final_ml_dataset
- где много нулей/пропусков (event-driven M2/M3, carry-forward M1/M5)

> Это lab-ноутбук: выводы здесь предварительные, не финальный отчёт. Меняй параметры в ячейке *Parameters* и перезапускай.

In [ ]:
# --- bootstrap: запуск из корня проекта (рядом с data/ и backend/) ---
import sys, os
from pathlib import Path
# найти корень проекта и встать в него
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'data' / 'processed').is_dir()), _here)
os.chdir(_root)
sys.path.insert(0, str(_root))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 60)
import importlib
from lab import utils as u
importlib.reload(u)   # подхватываем правки lab/utils.py без рестарта ядра
print('project root:', _root.name)

### Файлы в data/processed

In [ ]:
dd = u.data_dir()
files = sorted(dd.glob('*'))
inv = pd.DataFrame([{'file': f.name, 'size_kb': round(f.stat().st_size/1024,1)} for f in files])
inv.sort_values('size_kb', ascending=False).reset_index(drop=True)

### Основные датасеты: размеры и диапазоны дат

In [ ]:
rows = []
for name in ['final_ml_dataset.parquet','m1_features.parquet','m2_features.parquet',
             'm3_features.parquet','m4_features.parquet','m5_features.parquet']:
    p = dd / name
    if not p.exists():
        continue
    df = pd.read_parquet(p)
    dcol = 'date' if 'date' in df.columns else df.columns[0]
    dt = pd.to_datetime(df[dcol], errors='coerce')
    rows.append({'dataset': name, 'rows': len(df), 'cols': df.shape[1],
                 'date_min': dt.min(), 'date_max': dt.max()})
pd.DataFrame(rows)

### Final ML dataset — колонки по модулям

In [ ]:
d = u.load_final_dataset()
by_mod = {}
for c in d.columns:
    pref = c.split('_',1)[0]
    by_mod.setdefault(pref, []).append(c)
print('total columns:', d.shape[1])
for k in sorted(by_mod):
    print(f'{k:>6}: {len(by_mod[k])} cols')

### LSI whitelist (26 фич) — что присутствует в final_ml_dataset

In [ ]:
wl = u.get_lsi_whitelist()
av = u.available_whitelist(d)
missing = [f for f in wl if f not in av]
print(f'whitelist: {len(wl)}  present: {len(av)}  missing: {missing}')
u.split_features_by_module(av)

### Missingness / zero-rate по whitelist

In [ ]:
stats = u.compute_zero_null_stats(d, av)
stats

### Частота обновления: daily vs event-driven vs monthly/carry-forward
Высокий zero-rate у M2/M3 = event-driven (только в дни аукционов). M1/M5 MAD-scores меняются редко = monthly/carry-forward.

In [ ]:
def change_rate(s):
    s = pd.to_numeric(s, errors='coerce')
    return float((s.diff().abs() > 1e-9).mean())
freq = pd.DataFrame([{'feature': f, 'daily_change_rate': round(change_rate(d[f]),3),
                      'zero_rate': round(float((pd.to_numeric(d[f],errors='coerce')==0).mean()),3)}
                     for f in av])
freq['likely_cadence'] = np.where(freq['daily_change_rate']>0.5,'daily',
                          np.where(freq['daily_change_rate']>0.1,'frequent/event','monthly/carry-forward'))
freq.sort_values('daily_change_rate')

## Notes / Open questions

- M2/M3 zero-rate высокий — это норма (zero-fill в не-аукционные дни), не баг.
- m1_flag_end_of_period и m1_signal_final — проверить в 01/03.
- Какие raw-источники (ruonia.csv, keyrate.csv) дневные — пригодится для Local (06).